# Practical 1: Image Representation, Color Spaces & Preprocessing
**Course:** CSE471 — Deep Learning for Computer Vision  
**Unit:** Unit I (Introduction to Computer Vision & Image Representation) | **CO:** CO1  
**Problem Statement Context:** SIH26131 — Crop Foliar Image Preprocessing & Color Space Transformation  

---

## 1. Theoretical Foundations
In digital computer vision, an image is represented as a multidimensional tensor $I \in \mathbb{R}^{H 	imes W 	imes C}$:
- $H$: Height (rows)
- $W$: Width (columns)
- $C$: Channels (e.g., 3 for RGB/HSV/LAB, 1 for Grayscale)

### Color Spaces in Precision Agriculture:
1. **RGB (Red, Green, Blue):** Standard additive representation captured by CMOS sensor Bayer filters.
2. **Grayscale:** Luminance representation based on ITU-R BT.601 perceptual weighting:
   $$Y = 0.299R + 0.587G + 0.114B$$
3. **HSV (Hue, Saturation, Value):** Separates chromaticity (Hue: green leaf vs. yellow/brown lesion) from illumination variations (Value), making it robust to fluctuating sunlight in crop fields.
4. **LAB ($L^*, a^*, b^*$):** Perceptually uniform color space where $a^*$ represents the Green-Red axis and $b^*$ represents the Blue-Yellow axis.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Ensure repository root is on sys.path
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from src.transforms import convert_color_space

print(f"[✓] Root path: {REPO_ROOT}")

## 2. Load Agricultural Sample Images
We load a healthy leaf and an early blight diseased leaf with necrotic lesions.

In [ ]:
# Load samples from data/samples/
healthy_path = REPO_ROOT / "data" / "samples" / "classification" / "lab_plantvillage" / "Tomato___Healthy" / "sample_000.jpg"
diseased_path = REPO_ROOT / "data" / "samples" / "classification" / "lab_plantvillage" / "Tomato___Early_Blight" / "sample_000.jpg"

img_healthy = np.array(Image.open(healthy_path).convert("RGB"))
img_diseased = np.array(Image.open(diseased_path).convert("RGB"))

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(img_healthy)
axes[0].set_title("Healthy Tomato Leaf (RGB)")
axes[0].axis("off")

axes[1].imshow(img_diseased)
axes[1].set_title("Diseased Leaf: Early Blight (RGB)")
axes[1].axis("off")
plt.tight_layout()
plt.show()

## 3. Color Space Decomposition (RGB, Grayscale, HSV, LAB)
We analyze the diseased leaf across various color representations.

In [ ]:
gray = convert_color_space(img_diseased, "GRAYSCALE")
hsv = convert_color_space(img_diseased, "HSV")
lab = convert_color_space(img_diseased, "LAB")

fig, axes = plt.subplots(2, 2, figsize=(10, 10))
axes[0, 0].imshow(img_diseased)
axes[0, 0].set_title("Original RGB")
axes[0, 0].axis("off")

axes[0, 1].imshow(gray, cmap="gray")
axes[0, 1].set_title("Grayscale (Luminance)")
axes[0, 1].axis("off")

axes[1, 0].imshow(hsv)
axes[1, 0].set_title("HSV Color Space")
axes[1, 0].axis("off")

axes[1, 1].imshow(lab)
axes[1, 1].set_title("CIE LAB Color Space")
axes[1, 1].axis("off")

plt.tight_layout()
plt.show()

## 4. Color Channel Histograms: Healthy vs. Diseased Foliage
Notice the shift in the Green and Red channels due to necrotic lesion pigments and chlorophyll loss.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ('r', 'g', 'b')

for i, col in enumerate(colors):
    hist_h, _ = np.histogram(img_healthy[..., i], bins=64, range=(0, 256))
    hist_d, _ = np.histogram(img_diseased[..., i], bins=64, range=(0, 256))
    axes[0].plot(hist_h, color=col, label=f"Channel {col.upper()}")
    axes[1].plot(hist_d, color=col, label=f"Channel {col.upper()}")

axes[0].set_title("Healthy Leaf Pixel Intensity Distribution")
axes[0].set_xlabel("Pixel Value (0-255)")
axes[0].set_ylabel("Frequency")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_title("Diseased Leaf Pixel Intensity Distribution")
axes[1].set_xlabel("Pixel Value (0-255)")
axes[1].set_ylabel("Frequency")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Summary & Practical Observations
1. **Luminance vs. Chrominance:** Grayscale conversion captures general leaf geometry and structural veins but loses discriminatory chlorosis (yellow halo) signatures.
2. **HSV Utility:** Hue cleanly separates background clutter and necrotic lesions from viable chlorophyll green foliage.
3. **Implications for Deep CNNs:** Feeding 3-channel normalized RGB tensors allows CNN kernels to learn joint spatial-chromatic representations crucial for early disease classification.